# Week 4 — Retrieval-Augmented Generation (RAG)

**AI Agentic Engineering · Corte 1**

Companion notebook to `week-04-rag-content.html`. This is the last notebook before your Corte 1 project delivery
in Week 5 — the pipeline you build here is a working draft of that project.

**You will practice:**
1. Chunking a document with overlap.
2. Indexing chunks in ChromaDB, and the same vectors in FAISS for comparison.
3. A complete `answer_with_rag` function.
4. A light ADK agent with a `retrieve_context` tool, and the LangChain equivalent.
5. Two open exercises — including running this on your **own** project documents.


In [ ]:
%pip install -q --upgrade google-genai google-adk langchain-google-genai langgraph python-dotenv numpy chromadb faiss-cpu

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-flash-latest"
EMBED_MODEL = "gemini-embedding-001"

def embed(text):
    return client.models.embed_content(model=EMBED_MODEL, contents=text).embeddings[0].values

## 0. Sample corpus

A small self-contained knowledge base about this course, so the notebook runs without any external files. In the
lab, swap this for **your own project documents** (see Exercise 1).

In [ ]:
COURSE_DOCS = """
AI Agentic Engineering is a 16-week elective course for Systems Engineering students at Universidad de
Santander. It is organized into three graded cuts called cortes. Corte 1 (weeks 1-5) covers LLM fundamentals,
context engineering, and RAG. Corte 2 (weeks 6-11) covers agents, multi-agent systems, Google ADK, and
LangGraph. Corte 3 (weeks 12-16) covers evaluation, observability, and deployment to production.

Corte 1 is worth 30% of the final grade: 25% for a practical project and 5% for in-class activities such as
quizzes, workshops, and labs. The Corte 1 project requires building a conversational assistant that combines
context engineering and RAG over a set of documents chosen by the student, using the Gemini API and ChromaDB,
delivered as a GitHub repository with a live 10-minute demonstration.

Corte 2 is also worth 30%: 25% for a multi-agent system project and 5% for in-class activities. Students must
build a multi-agent system that solves a real problem using Google ADK or LangGraph, integrating RAG
capabilities, delivered with a system diagram and a 15-minute live demonstration.

Corte 3 is worth 40% of the grade: 30% for a final integrator project and 10% for in-class activities. The
final project must combine agents, RAG, an automatic evaluation pipeline, tracing with Langfuse, and a REST
API exposed with FastAPI, delivered with a 5-minute demo video and a 20-minute technical presentation.

All labs in this course use free-tier tools: the Gemini API through Google AI Studio, ChromaDB and FAISS for
vector storage, and Langfuse's free tier for observability starting in Corte 3. Students are also introduced,
at a basic level, to Google Antigravity CLI, a terminal-based coding agent that can scaffold and edit project
files.

Class time each week is split into two blocks: two hours of theory with instructor-led demonstrations and
class discussion, followed by four hours of hands-on lab. Labs mix guided coding, small-group workshops,
individual work, and peer code review. All lab work is submitted through the course GitHub repository, with
Moodle used for supporting material and announcements.
"""

## 1. Chunking

In [ ]:
def chunk_text(text: str, chunk_size: int = 400, overlap: int = 60) -> list[str]:
    text = " ".join(text.split())  # normalize whitespace
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

chunks = chunk_text(COURSE_DOCS, chunk_size=400, overlap=60)
print(f"{len(COURSE_DOCS)} chars -> {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(f"[{i}] {c[:70]}...")

## 2. Indexing with ChromaDB

In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db_week4")
# start fresh each run of this notebook
try:
    chroma_client.delete_collection("course_docs")
except Exception:
    pass
collection = chroma_client.create_collection(name="course_docs")

collection.add(
    documents=chunks,
    embeddings=[embed(c) for c in chunks],
    ids=[f"chunk-{i}" for i in range(len(chunks))],
)
print(collection.count(), "chunks indexed")

### 2a. The same vectors in FAISS (for comparison)

In [ ]:
import faiss

vectors = np.array([embed(c) for c in chunks], dtype="float32")
faiss_index = faiss.IndexFlatL2(vectors.shape[1])
faiss_index.add(vectors)

query_vector = np.array([embed("How is Corte 1 graded?")], dtype="float32")
distances, indices = faiss_index.search(query_vector, k=3)
for dist, idx in zip(distances[0], indices[0]):
    print(f"{dist:.3f}  {chunks[idx][:80]}...")

## 3. A complete `answer_with_rag` function

In [ ]:
def answer_with_rag(question: str, n_results: int = 3, verbose: bool = False) -> str:
    results = collection.query(query_embeddings=[embed(question)], n_results=n_results)
    retrieved_chunks = results["documents"][0]
    if verbose:
        print("--- retrieved chunks ---")
        for c in retrieved_chunks:
            print("-", c[:80], "...")
    context = "\n\n---\n\n".join(retrieved_chunks)

    prompt = f'''Answer the question using ONLY the context below. If the answer isn't
in the context, say you don't have that information — do not make anything up.

Context:
{context}

Question: {question}
Answer:'''

    response = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return response.text

print(answer_with_rag("How is Corte 1 graded?", verbose=True))
print()
print(answer_with_rag("What programming language does the course use?"))  # not in the docs -> should decline

## 4. Light preview: RAG as an ADK tool, and the LangChain equivalent

We're one step away from a real agent (Week 6 onward covers tool use and ReAct properly). For now, notice that
"retrieval" is just a Python function — which is exactly what an ADK **tool** is.

In [ ]:
import asyncio
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

def retrieve_context(query: str) -> dict:
    """Retrieve the most relevant course-document chunks for a query."""
    results = collection.query(query_embeddings=[embed(query)], n_results=3)
    return {"chunks": results["documents"][0]}

rag_agent = Agent(
    model=MODEL,
    name="course_rag_agent",
    instruction=(
        "Use the retrieve_context tool to find relevant chunks before answering. "
        "Answer ONLY using information returned by the tool. If it's not there, say so."
    ),
    tools=[retrieve_context],
)

def ask_adk_agent(agent, prompt, app_name="week4_app", user_id="student"):
    runner = InMemoryRunner(agent=agent, app_name=app_name)
    session = asyncio.run(runner.session_service.create_session(app_name=app_name, user_id=user_id))
    content = types.Content(role="user", parts=[types.Part.from_text(text=prompt)])
    final_text = None
    for event in runner.run(user_id=user_id, session_id=session.id, new_message=content):
        if event.content and event.content.parts and event.content.parts[0].text:
            final_text = event.content.parts[0].text
    return final_text

print(ask_adk_agent(rag_agent, "How is Corte 2 graded?"))

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model=MODEL)

def answer_with_rag_langchain(question: str, n_results: int = 3) -> str:
    results = collection.query(query_embeddings=[embed(question)], n_results=n_results)
    context = "\n\n---\n\n".join(results["documents"][0])
    prompt = (
        "Answer the question using ONLY the context below. If the answer isn't in the "
        "context, say you don't have that information.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    return llm.invoke(prompt).content

print(answer_with_rag_langchain("What tools are used for observability in this course?"))

## 5. Exercises

In [ ]:
# TODO Exercise 1 — Run this on YOUR OWN Corte 1 project documents
# 1. Put the plain-text documents you gathered in Week 3 into a `data/` folder next to this notebook.
# 2. Load and concatenate them (or keep them as separate documents with metadata — see Exercise 2).
# 3. Re-run chunking, indexing, and answer_with_rag with 5 real questions a user of your assistant would ask.

# your code here


In [ ]:
# TODO Exercise 2 — Metadata filtering
# Re-index the chunks, but this time attach a `metadatas=[{"source": "..."}, ...]` list to
# collection.add() so each chunk remembers which source document it came from.
# Then query with a `where={"source": "..."}` filter to restrict retrieval to just one document.
# (This is a preview of "RAG avanzado: filtrado por metadata", covered in depth in Week 10.)

# your code here


## Corte 1 wrap-up

You now have every piece needed for your Week 5 project: a system prompt (Week 2), context/history handling
(Week 3), and a working RAG pipeline (this week). See `week-04-rag-activities.html` for the full delivery
checklist and grading rubric.

## Looking ahead

Corte 2 starts in Week 6 with agent fundamentals — tool use, function calling, and the ReAct pattern — building
directly on the `tools=[retrieve_context]` preview above.